# Juvenile Mortality Modeling Lab

This notebook is set up as a practical actuarial lab for **juvenile mortality modeling**.  
The intent is not just to fit models, but to produce results that are stable enough to compare, explain, and carry into a report.

It is organized into four sessions:

1. **Data overview and scope**
2. **Feature screening**
3. **Mortality model fitting and comparison**
4. **Validation and sheet-style mortality output**

### Working style
This notebook still avoids heavy automation.  
You can change filters, formulas, feature sets, or comment out blocks without breaking the lab structure.

In [13]:
from __future__ import annotations

import warnings
from pathlib import Path
import math
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display
from patsy.builtins import bs

warnings.filterwarnings("default")
warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

try:
    import duckdb
    HAS_DUCKDB = True
except Exception:
    HAS_DUCKDB = False

try:
    import pyarrow  # noqa: F401
    HAS_PYARROW = True
except Exception:
    HAS_PYARROW = False

try:
    from sklearn.ensemble import (
        ExtraTreesRegressor,
        GradientBoostingRegressor,
        HistGradientBoostingRegressor,
        RandomForestRegressor,
    )
    from sklearn.inspection import permutation_importance
    from sklearn.linear_model import PoissonRegressor
    HAS_SKLEARN = True
except Exception:
    HAS_SKLEARN = False

print("duckdb available :", HAS_DUCKDB)
print("pyarrow available:", HAS_PYARROW)
print("sklearn available:", HAS_SKLEARN)

duckdb available : True
pyarrow available: True
sklearn available: True


---

# Session 1 · Data overview and filters

Edit the control panel first. Then rerun the notebook from Session 1 onward.

In [28]:
# ============================================================
# CONTROL PANEL — EDIT HERE FIRST
# ============================================================

DATA_PATH = Path("script/data/parquet/full.parquet")

# ------------------------------
# Core scope filters
# ------------------------------
OBSERVATION_YEARS = None          # Example: [2018, 2019]; use None to keep all years
ISSUE_AGE_MIN = 0
ISSUE_AGE_MAX = 17
ATTAINED_AGE_MIN = 1           # Example: 0 or 5; None keeps all lower ages
ATTAINED_AGE_MAX = 94

AGE_IND_KEEP = None               # Example: ["0"] or ["1"]; None keeps all age bases
SEX_KEEP = None                   # Example: ["M"] or ["F"]; None keeps all sexes
SMOKER_STATUS_KEEP = ["U"]        # Example: ["U"]; set to None to keep all
INSURANCE_PLAN_DROP = ["Other"]   # Set to [] if you want to keep "Other"

EXCLUDE_POST_LEVEL_TERM = True
POST_LEVEL_TERM_EXCLUDE_VALUES = ["PLT"]

# Optional duration cleanup for summary work only.
# Keep FALSE if you want sheet-style outputs similar to the reference tables.
COLLAPSE_DURATION_26_PLUS = False

# ------------------------------
# 1M face-amount cap option
# ------------------------------
CAP_FACE_AT_1M = True
FACE_CAP_VALUE = 1_000_000
FACE_CAP_SOURCE_BANDS = [
    "08: 1,000,000 - 2,499,999",
    "09: 2,500,000 - 4,999,999",
    "10: 5,000,000 - 9,999,999",
    "11: 10,000,000+",
]
FACE_CAP_TARGET_BAND = "08: 1,000,000+"

# ------------------------------
# Metric columns
# ------------------------------
ACTUAL_CNT_COL = "Death_Count"
EXPECTED_CNT_COL = "ExpDth_VBT2015_Cnt"
ACTUAL_AMT_COL = "Death_Claim_Amount"
EXPECTED_AMT_COL = "ExpDth_VBT2015_Amt"

NUMERIC_COLS = [
    "Observation_Year", "Issue_Age", "Duration", "Issue_Year", "Attained_Age",
    "Amount_Exposed", "Policies_Exposed", ACTUAL_AMT_COL, ACTUAL_CNT_COL,
    EXPECTED_CNT_COL, EXPECTED_AMT_COL
]

CATEGORICAL_COLS = [
    "Age_Ind", "Sex", "Smoker_Status", "Insurance_Plan", "Face_Amount_Band",
    "SOA_Antp_Lvl_TP", "SOA_Guar_Lvl_TP", "SOA_Post_Lvl_Ind", "Slct_Ult_Ind",
    "Preferred_Indicator", "Number_of_Pfd_Classes", "Preferred_Class"
]

BASE_COLUMNS = NUMERIC_COLS + CATEGORICAL_COLS

# ------------------------------
# Session 2: screening controls
# ------------------------------
SCREEN_FEATURES = [
    "Age_Ind",
    "Sex",
    "Insurance_Plan",
    "Face_Amount_Band",
    "Slct_Ult_Ind",
    "Preferred_Indicator",
    "Number_of_Pfd_Classes",
    "Issue_Age",
    "Attained_Age",
    "Duration",
    "Observation_Year",
    "SOA_Guar_Lvl_TP",
    "SOA_Antp_Lvl_TP",
    "SOA_Post_Lvl_Ind",
]
NUMERIC_SCREEN_FEATURES = ["Issue_Age", "Attained_Age", "Duration", "Observation_Year"]
SCREEN_SPLINE_DF = 6
SCREEN_TARGET_SMOOTH = 0.25
SCREEN_RANDOM_STATE = 42

# Manual shortlist after screening.
SHORTLIST_FEATURES = [
    "Age_Ind",
    "Sex",
    "Insurance_Plan",
    "Face_Amount_Band",
    "Slct_Ult_Ind",
    "Issue_Age",
    "Duration",
]

# ------------------------------
# Session 3: model comparison controls
# ------------------------------
HOLDOUT_YEARS = None

GLM_FORMULA_CORE = (
    f"{ACTUAL_CNT_COL} ~ C(Age_Ind) + C(Sex) + C(Insurance_Plan) + C(Face_Amount_Band) "
    f"+ C(Slct_Ult_Ind) "
    f"+ bs(Issue_Age, df=6, degree=3, include_intercept=False) "
    f"+ bs(Duration, df=6, degree=3, include_intercept=False)"
)

GLM_FORMULA_ATTAINED = (
    f"{ACTUAL_CNT_COL} ~ C(Age_Ind) + C(Sex) + C(Insurance_Plan) + C(Face_Amount_Band) "
    f"+ C(Slct_Ult_Ind) "
    f"+ bs(Attained_Age, df=7, degree=3, include_intercept=False) "
    f"+ bs(Duration, df=6, degree=3, include_intercept=False)"
)

ML_FEATURES = SHORTLIST_FEATURES.copy()
MODEL_TARGET_SMOOTH = 0.25
MODEL_RANDOM_STATE = 42

# ------------------------------
# Session 4: validation / review controls
# ------------------------------
SELECTED_MODEL_KEY = "poisson_glm_core"
SEGMENT_TOLERANCE = 0.10
MIN_EXPECTED_FOR_REVIEW = 3.0

# ------------------------------
# Sheet-style mortality output
# These settings let you produce one sheet-like result at a time,
# similar in shape to the reference Excel tables.
# ------------------------------
OUTPUT_AGE_IND = "0"              # "0" = ANB, "1" = ALB
OUTPUT_SEX = "M"                  # "M" or "F"
OUTPUT_SELECT_PERIOD = 25
OUTPUT_ULT_DURATION = 26
OUTPUT_DURATION_COLUMNS = list(range(1, 26))
OUTPUT_RATE_PER = 1000
OUTPUT_ISSUE_AGE_MIN = 0
OUTPUT_ISSUE_AGE_MAX = 17

# Optional benchmark comparison
BENCHMARK_XLSX_PATH = None        # Example: Path("2015-vbt-unismoke-alb-anb.xlsx")
BENCHMARK_SHEET_NAME = None       # Example: "2015 Male Unismoke ANB"

EXPORT_DIR = Path("script/model_outputs")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:

# ============================================================
# DATA DICTIONARY USED BY THIS NOTEBOOK
# ============================================================

DATA_DICTIONARY = {
    "Observation_Year": "Calendar year of observation (2012-2019).",
    "Age_Ind": "0 = ANB, 1 = ALB, 2 = Age next birthday.",
    "Sex": "F = Female, M = Male.",
    "Smoker_Status": "N = Nonsmoker, S = Smoker, U = Uni-smoke.",
    "Insurance_Plan": "Perm, Term, UL, ULSG, VL, VLSG, Other.",
    "Issue_Age": "Issue age.",
    "Duration": "Policy duration.",
    "Face_Amount_Band": "Face amount band (01 to 11).",
    "Issue_Year": "Issue year.",
    "Attained_Age": "Attained age.",
    "SOA_Antp_Lvl_TP": "SOA anticipated level term period.",
    "SOA_Guar_Lvl_TP": "SOA guaranteed level term period.",
    "SOA_Post_Lvl_Ind": "SOA post level term indicator.",
    "Slct_Ult_Ind": "S = Select, U = Ultimate.",
    "Preferred_Indicator": "0 = Not preferred, 1 = Preferred, U = Unknown.",
    "Number_of_Pfd_Classes": "Number of preferred classes.",
    "Preferred_Class": "Preferred class code.",
    "Amount_Exposed": "Amount exposed.",
    "Policies_Exposed": "Policies exposed.",
    "Death_Claim_Amount": "Observed death claim amount.",
    "Death_Count": "Observed death count.",
    "ExpDth_VBT2015_Cnt": "Expected deaths on count basis under 2015 VBT.",
    "ExpDth_VBT2015_Amt": "Expected deaths on amount basis under 2015 VBT.",
}

display(pd.DataFrame(
    {"Column": list(DATA_DICTIONARY.keys()), "Description": list(DATA_DICTIONARY.values())}
))


,Column,Description
0,Observation_Year,Calendar year of observation (2012-2019).
1,Age_Ind,"0 = ANB, 1 = ALB, 2 = Age next birthday."
2,Sex,"F = Female, M = Male."
3,Smoker_Status,"N = Nonsmoker, S = Smoker, U = Uni-smoke."
4,Insurance_Plan,"Perm, Term, UL, ULSG, VL, VLSG, Other."
5,Issue_Age,Issue age.
6,Duration,Policy duration.
7,Face_Amount_Band,Face amount band (01 to 11).
8,Issue_Year,Issue year.
9,Attained_Age,Attained age.


In [15]:
# ============================================================
# HELPER FUNCTIONS
# ============================================================

def require_parquet_support():
    if HAS_DUCKDB or HAS_PYARROW:
        return
    raise ImportError(
        "This notebook needs either duckdb or pyarrow to read parquet. "
        "Install one of them locally, for example: pip install duckdb or pip install pyarrow"
    )


def require_sklearn():
    if HAS_SKLEARN:
        return
    raise ImportError(
        "This notebook needs scikit-learn for Sessions 2 and 3. "
        "Install it locally, for example: pip install scikit-learn"
    )


def read_parquet_data(path: Path, columns: list[str] | None = None) -> pd.DataFrame:
    require_parquet_support()

    if not path.exists():
        raise FileNotFoundError(f"Parquet file not found: {path}")

    if HAS_DUCKDB:
        cols_sql = "*" if columns is None else ", ".join([f'"{c}"' for c in columns])
        query = f"SELECT {cols_sql} FROM read_parquet('{path.as_posix()}')"
        return duckdb.sql(query).df()

    return pd.read_parquet(path, columns=columns)


def export_csv(df: pd.DataFrame, filename: str):
    path = EXPORT_DIR / filename
    df.to_csv(path, index=False)
    print(f"Saved: {path}")


def safe_divide(numerator, denominator):
    numerator = np.asarray(numerator, dtype=float)
    denominator = np.asarray(denominator, dtype=float)
    out = np.divide(
        numerator,
        denominator,
        out=np.full_like(numerator, np.nan, dtype=float),
        where=np.abs(denominator) > 1e-12,
    )
    if np.ndim(out) == 0:
        return float(out)
    if out.size == 1:
        return float(np.ravel(out)[0])
    return out


def standardize_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in NUMERIC_COLS:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    for col in CATEGORICAL_COLS:
        if col in out.columns:
            out[col] = out[col].astype("string").fillna("Missing")

    return out


def apply_face_amount_cap_1m(df: pd.DataFrame) -> pd.DataFrame:
    # Requested cap rule:
    # - bands 08/09/10/11 -> '08: 1,000,000+'
    # - Death_Claim_Amount = 1,000,000 * Death_Count
    # - Amount_Exposed     = 1,000,000 * Policies_Exposed
    # - ExpDth_VBT2015_Amt = 1,000,000 * ExpDth_VBT2015_Cnt
    # The last line keeps amount-based A/E internally consistent after the cap.
    if not CAP_FACE_AT_1M:
        return df.copy()

    out = df.copy()
    mask = out["Face_Amount_Band"].isin(FACE_CAP_SOURCE_BANDS)

    out.loc[mask, "Face_Amount_Band"] = FACE_CAP_TARGET_BAND
    out.loc[mask, ACTUAL_AMT_COL] = FACE_CAP_VALUE * out.loc[mask, ACTUAL_CNT_COL]
    out.loc[mask, "Amount_Exposed"] = FACE_CAP_VALUE * out.loc[mask, "Policies_Exposed"]
    out.loc[mask, EXPECTED_AMT_COL] = FACE_CAP_VALUE * out.loc[mask, EXPECTED_CNT_COL]

    return out


def apply_scope_filters(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = standardize_frame(df)
    steps = [{"step": "raw_input", "rows": len(out)}]

    if OBSERVATION_YEARS is not None:
        out = out[out["Observation_Year"].isin(OBSERVATION_YEARS)].copy()
        steps.append({"step": f"Observation_Year in {OBSERVATION_YEARS}", "rows": len(out)})

    out = out[
        out["Issue_Age"].between(ISSUE_AGE_MIN, ISSUE_AGE_MAX, inclusive="both")
    ].copy()
    steps.append({"step": f"Issue_Age between {ISSUE_AGE_MIN} and {ISSUE_AGE_MAX}", "rows": len(out)})

    if ATTAINED_AGE_MIN is not None:
        out = out[out["Attained_Age"] >= ATTAINED_AGE_MIN].copy()
        steps.append({"step": f"Attained_Age >= {ATTAINED_AGE_MIN}", "rows": len(out)})

    if ATTAINED_AGE_MAX is not None:
        out = out[out["Attained_Age"] <= ATTAINED_AGE_MAX].copy()
        steps.append({"step": f"Attained_Age <= {ATTAINED_AGE_MAX}", "rows": len(out)})

    if AGE_IND_KEEP is not None:
        out = out[out["Age_Ind"].isin(AGE_IND_KEEP)].copy()
        steps.append({"step": f"Age_Ind in {AGE_IND_KEEP}", "rows": len(out)})

    if SEX_KEEP is not None:
        out = out[out["Sex"].isin(SEX_KEEP)].copy()
        steps.append({"step": f"Sex in {SEX_KEEP}", "rows": len(out)})

    if SMOKER_STATUS_KEEP is not None:
        out = out[out["Smoker_Status"].isin(SMOKER_STATUS_KEEP)].copy()
        steps.append({"step": f"Smoker_Status in {SMOKER_STATUS_KEEP}", "rows": len(out)})

    if INSURANCE_PLAN_DROP:
        out = out[~out["Insurance_Plan"].isin(INSURANCE_PLAN_DROP)].copy()
        steps.append({"step": f"Insurance_Plan not in {INSURANCE_PLAN_DROP}", "rows": len(out)})

    if EXCLUDE_POST_LEVEL_TERM:
        out = out[~out["SOA_Post_Lvl_Ind"].isin(POST_LEVEL_TERM_EXCLUDE_VALUES)].copy()
        steps.append({"step": f"SOA_Post_Lvl_Ind not in {POST_LEVEL_TERM_EXCLUDE_VALUES}", "rows": len(out)})

    if COLLAPSE_DURATION_26_PLUS:
        out["Duration"] = np.where(out["Duration"] >= 26, 26, out["Duration"])
        steps.append({"step": "Duration >= 26 collapsed to 26", "rows": len(out)})

    out = out[out[EXPECTED_CNT_COL] > 0].copy()
    steps.append({"step": f"{EXPECTED_CNT_COL} > 0", "rows": len(out)})

    out = apply_face_amount_cap_1m(out)
    if CAP_FACE_AT_1M:
        steps.append({"step": "Applied 1M face cap rule", "rows": len(out)})

    return out.reset_index(drop=True), pd.DataFrame(steps)


def overall_metrics(df: pd.DataFrame) -> pd.DataFrame:
    actual_cnt = float(df[ACTUAL_CNT_COL].sum())
    expected_cnt = float(df[EXPECTED_CNT_COL].sum())
    actual_amt = float(df[ACTUAL_AMT_COL].sum())
    expected_amt = float(df[EXPECTED_AMT_COL].sum())
    policies = float(df["Policies_Exposed"].sum())
    amount_exposed = float(df["Amount_Exposed"].sum())

    return pd.DataFrame({
        "Metric": [
            "Rows",
            "Policies Exposed",
            "Amount Exposed",
            "Death Count",
            "Expected Death Count",
            "A/E Count",
            "Death Claim Amount",
            "Expected Death Amount",
            "A/E Amount",
            "Actual Rate per 1000",
            "Expected Rate per 1000",
        ],
        "Value": [
            len(df),
            policies,
            amount_exposed,
            actual_cnt,
            expected_cnt,
            safe_divide(actual_cnt, expected_cnt),
            actual_amt,
            expected_amt,
            safe_divide(actual_amt, expected_amt),
            safe_divide(actual_cnt * 1000.0, policies),
            safe_divide(expected_cnt * 1000.0, policies),
        ]
    })


def ae_summary(df: pd.DataFrame, group_cols: str | list[str]) -> pd.DataFrame:
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    out = (
        df.groupby(group_cols, dropna=False)[
            [ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL, "Policies_Exposed", "Amount_Exposed"]
        ]
        .sum()
        .reset_index()
    )
    out["A/E_Count"] = safe_divide(out[ACTUAL_CNT_COL], out[EXPECTED_CNT_COL])
    out["A/E_Amount"] = safe_divide(out[ACTUAL_AMT_COL], out[EXPECTED_AMT_COL])
    out["Actual_Rate_per_1000"] = safe_divide(out[ACTUAL_CNT_COL] * 1000.0, out["Policies_Exposed"])
    out["Expected_Rate_per_1000"] = safe_divide(out[EXPECTED_CNT_COL] * 1000.0, out["Policies_Exposed"])
    return out.sort_values(group_cols).reset_index(drop=True)


def safe_poisson_deviance(y_true, y_pred) -> float:
    y = np.asarray(y_true, dtype=float)
    mu = np.clip(np.asarray(y_pred, dtype=float), 1e-12, None)
    term = np.where(y == 0, 0.0, y * np.log(np.clip(y, 1e-12, None) / mu))
    return float(2.0 * np.sum(term - (y - mu)))


def weighted_ae_error(actual, predicted, expected) -> float:
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    expected = np.asarray(expected, dtype=float)

    actual_ae = actual / np.clip(expected, 1e-12, None)
    pred_ae = predicted / np.clip(expected, 1e-12, None)
    return float(np.average(np.abs(actual_ae - pred_ae), weights=np.clip(expected, 1e-12, None)))


def prepare_model_frame(df: pd.DataFrame, required_cols: list[str]) -> pd.DataFrame:
    required_cols = list(dict.fromkeys(required_cols))
    out = df[required_cols].copy()

    for col in required_cols:
        if col in CATEGORICAL_COLS:
            out[col] = out[col].astype("string").fillna("Missing")
        elif col in NUMERIC_COLS:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    out = out.replace([np.inf, -np.inf], np.nan)
    keep_cols = [c for c in required_cols if c not in [ACTUAL_AMT_COL, EXPECTED_AMT_COL]]
    out = out.dropna(subset=keep_cols).copy()
    out = out[out[EXPECTED_CNT_COL] > 0].copy()
    return out.reset_index(drop=True)


def aggregate_cells(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    metric_cols = [
        ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL,
        "Policies_Exposed", "Amount_Exposed"
    ]
    metric_cols = [c for c in metric_cols if c in df.columns]

    if not features:
        agg = df[metric_cols].sum().to_frame().T
        return agg.reset_index(drop=True)

    agg = (
        df.groupby(features, dropna=False)[metric_cols]
        .sum()
        .reset_index()
        .sort_values(features)
        .reset_index(drop=True)
    )
    return agg


def split_feature_types(features: list[str]) -> tuple[list[str], list[str]]:
    cat = [f for f in features if f in CATEGORICAL_COLS]
    num = [f for f in features if f in NUMERIC_COLS and f not in cat]
    return cat, num


def build_category_levels(df: pd.DataFrame, features: list[str]) -> dict[str, list[str]]:
    cat_cols, _ = split_feature_types(features)
    levels = {}
    for col in cat_cols:
        levels[col] = sorted(df[col].astype("string").fillna("Missing").unique().tolist())
    return levels


def encode_features(
    df: pd.DataFrame,
    features: list[str],
    category_levels: dict[str, list[str]] | None = None,
    design_columns: list[str] | None = None,
) -> pd.DataFrame:
    work = df[features].copy()
    cat_cols, num_cols = split_feature_types(features)

    for col in cat_cols:
        work[col] = work[col].astype("string").fillna("Missing")
        if category_levels is not None and col in category_levels:
            work[col] = pd.Categorical(work[col], categories=category_levels[col])

    enc = pd.get_dummies(work, columns=cat_cols, drop_first=False, dtype=float)

    for col in num_cols:
        enc[col] = pd.to_numeric(enc[col], errors="coerce")

    enc = enc.replace([np.inf, -np.inf], np.nan).fillna(0.0)

    if design_columns is not None:
        for col in design_columns:
            if col not in enc.columns:
                enc[col] = 0.0
        enc = enc[design_columns].copy()

    return enc.astype(float)


def evaluate_predictions(actual, expected, predicted, label: str) -> dict:
    actual = np.asarray(actual, dtype=float)
    expected = np.asarray(expected, dtype=float)
    predicted = np.clip(np.asarray(predicted, dtype=float), 1e-12, None)

    return {
        "Dataset": label,
        "Rows": len(actual),
        "Actual_Deaths": float(actual.sum()),
        "Expected_VBT": float(expected.sum()),
        "Predicted_Deaths": float(predicted.sum()),
        "Actual_to_Predicted": safe_divide(actual.sum(), predicted.sum()),
        "Actual_to_VBT": safe_divide(actual.sum(), expected.sum()),
        "Poisson_Deviance": safe_poisson_deviance(actual, predicted),
        "Weighted_AE_Error": weighted_ae_error(actual, predicted, expected),
    }


def screen_formula_for_feature(feature: str) -> str:
    if feature in NUMERIC_SCREEN_FEATURES:
        return f"{ACTUAL_CNT_COL} ~ bs({feature}, df={SCREEN_SPLINE_DF}, degree=3, include_intercept=False)"
    return f"{ACTUAL_CNT_COL} ~ C({feature})"


def run_univariate_screen(df: pd.DataFrame, features: list[str]) -> pd.DataFrame:
    rows = []

    for feature in features:
        needed = [feature, ACTUAL_CNT_COL, EXPECTED_CNT_COL]
        sub = prepare_model_frame(df, needed)

        try:
            sub = aggregate_cells(sub, [feature])

            base_model = smf.glm(
                formula=f"{ACTUAL_CNT_COL} ~ 1",
                data=sub,
                family=sm.families.Poisson(),
                offset=np.log(sub[EXPECTED_CNT_COL]),
            ).fit()

            model = smf.glm(
                formula=screen_formula_for_feature(feature),
                data=sub,
                family=sm.families.Poisson(),
                offset=np.log(sub[EXPECTED_CNT_COL]),
            ).fit()

            pred = model.predict(sub, offset=np.log(sub[EXPECTED_CNT_COL]))
            rows.append({
                "Feature": feature,
                "Levels_or_Unique": int(sub[feature].nunique(dropna=False)),
                "Base_Deviance": base_model.deviance,
                "Model_Deviance": model.deviance,
                "Deviance_Reduction": base_model.deviance - model.deviance,
                "Deviance_Reduction_%": 100 * (base_model.deviance - model.deviance) / base_model.deviance if base_model.deviance > 0 else np.nan,
                "AIC": model.aic,
                "Weighted_AE_Error": weighted_ae_error(sub[ACTUAL_CNT_COL], pred, sub[EXPECTED_CNT_COL]),
            })
        except Exception as exc:
            rows.append({
                "Feature": feature,
                "Levels_or_Unique": np.nan,
                "Base_Deviance": np.nan,
                "Model_Deviance": np.nan,
                "Deviance_Reduction": np.nan,
                "Deviance_Reduction_%": np.nan,
                "AIC": np.nan,
                "Weighted_AE_Error": np.nan,
                "Note": str(exc)[:200],
            })

    out = pd.DataFrame(rows)
    return out.sort_values(["Deviance_Reduction_%", "Deviance_Reduction"], ascending=False, na_position="last").reset_index(drop=True)


def run_tree_feature_importance(
    df: pd.DataFrame,
    features: list[str],
    estimator,
    smooth: float = 0.25,
    random_state: int = 42,
) -> pd.DataFrame:
    require_sklearn()

    needed = features + [ACTUAL_CNT_COL, EXPECTED_CNT_COL]
    base = prepare_model_frame(df, needed)
    cell_df = aggregate_cells(base, features).copy()

    category_levels = build_category_levels(cell_df, features)
    X = encode_features(cell_df, features, category_levels=category_levels)
    y_log_ratio = np.log((cell_df[ACTUAL_CNT_COL] + smooth) / (cell_df[EXPECTED_CNT_COL] + smooth))
    w = np.clip(cell_df[EXPECTED_CNT_COL].to_numpy(dtype=float), 1e-12, None)

    estimator.fit(X, y_log_ratio, sample_weight=w)

    if hasattr(estimator, "feature_importances_"):
        raw_imp = pd.Series(estimator.feature_importances_, index=X.columns)
    else:
        perm = permutation_importance(
            estimator, X, y_log_ratio, n_repeats=10, random_state=random_state,
            scoring="neg_mean_absolute_error"
        )
        raw_imp = pd.Series(perm.importances_mean, index=X.columns)

    rows = []
    for feature in features:
        mask = raw_imp.index == feature
        mask = mask | raw_imp.index.str.startswith(f"{feature}_")
        rows.append({
            "Feature": feature,
            "Importance": float(raw_imp[mask].sum()),
        })

    out = pd.DataFrame(rows).sort_values("Importance", ascending=False).reset_index(drop=True)
    out["Rank"] = np.arange(1, len(out) + 1)
    return out


def combine_screen_ranks(
    univariate_df: pd.DataFrame | None = None,
    gbm_df: pd.DataFrame | None = None,
    extra_df: pd.DataFrame | None = None,
) -> pd.DataFrame:
    pieces = []

    if univariate_df is not None:
        uni = univariate_df[["Feature", "Deviance_Reduction_%"]].copy()
        uni["Univariate_Rank"] = uni["Deviance_Reduction_%"].rank(ascending=False, method="dense")
        pieces.append(uni)

    if gbm_df is not None:
        gbm = gbm_df[["Feature", "Importance"]].rename(columns={"Importance": "GBM_Importance"})
        gbm["GBM_Rank"] = gbm["GBM_Importance"].rank(ascending=False, method="dense")
        pieces.append(gbm)

    if extra_df is not None:
        extra = extra_df[["Feature", "Importance"]].rename(columns={"Importance": "ExtraTrees_Importance"})
        extra["ExtraTrees_Rank"] = extra["ExtraTrees_Importance"].rank(ascending=False, method="dense")
        pieces.append(extra)

    if not pieces:
        raise ValueError("No screening result tables were provided.")

    out = pieces[0].copy()
    for piece in pieces[1:]:
        out = out.merge(piece, on="Feature", how="outer")

    rank_cols = [c for c in out.columns if c.endswith("_Rank")]
    sort_anchor = "Deviance_Reduction_%" if "Deviance_Reduction_%" in out.columns else rank_cols[0]
    out["Average_Rank"] = out[rank_cols].mean(axis=1)
    out = out.sort_values(["Average_Rank", sort_anchor], ascending=[True, False], na_position="last").reset_index(drop=True)
    return out


def extract_formula_features(formula: str) -> list[str]:
    tokens = set(re.findall(r"\b[A-Za-z_][A-Za-z0-9_]*\b", formula))
    metric_exclusions = {
        ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL,
        "Policies_Exposed", "Amount_Exposed"
    }
    return [c for c in BASE_COLUMNS if c in tokens and c not in metric_exclusions]


def score_with_bundle(bundle: dict, score_df: pd.DataFrame) -> pd.DataFrame:
    validation_keep_cols = [
        "Observation_Year", "Age_Ind", "Sex", "Insurance_Plan", "Face_Amount_Band",
        "Issue_Age", "Attained_Age", "Duration", "Slct_Ult_Ind",
        "Policies_Exposed", "Amount_Exposed", ACTUAL_CNT_COL, EXPECTED_CNT_COL,
        ACTUAL_AMT_COL, EXPECTED_AMT_COL
    ]
    validation_keep_cols = [c for c in validation_keep_cols if c in score_df.columns]

    features = bundle["features"]
    needed = list(dict.fromkeys(features + validation_keep_cols))
    score_base = prepare_model_frame(score_df, needed)

    if bundle["model_type"] == "statsmodels_glm":
        pred = bundle["result"].predict(score_base, offset=np.log(score_base[EXPECTED_CNT_COL]))
    else:
        X_score = encode_features(
            score_base,
            features,
            category_levels=bundle["category_levels"],
            design_columns=bundle["design_columns"],
        )

        if bundle["model_type"] == "poisson_regressor":
            pred_rate = np.clip(bundle["result"].predict(X_score), 1e-12, None)
            pred = pred_rate * score_base[EXPECTED_CNT_COL].to_numpy(dtype=float)
        elif bundle["model_type"] == "log_ratio_model":
            pred_log = bundle["result"].predict(X_score)
            pred = np.exp(pred_log) * score_base[EXPECTED_CNT_COL].to_numpy(dtype=float)
        else:
            raise ValueError(f"Unsupported model_type: {bundle['model_type']}")

    out = score_base.copy()
    out["Predicted_Deaths"] = np.clip(np.asarray(pred, dtype=float), 1e-12, None)
    out["Predicted_to_VBT"] = safe_divide(out["Predicted_Deaths"], out[EXPECTED_CNT_COL])
    out["Actual_to_Predicted"] = safe_divide(out[ACTUAL_CNT_COL], out["Predicted_Deaths"])
    out["Actual_Rate_per_1000"] = safe_divide(out[ACTUAL_CNT_COL] * 1000.0, out["Policies_Exposed"])
    out["Expected_Rate_per_1000"] = safe_divide(out[EXPECTED_CNT_COL] * 1000.0, out["Policies_Exposed"])
    out["Predicted_Rate_per_1000"] = safe_divide(out["Predicted_Deaths"] * 1000.0, out["Policies_Exposed"])
    return out


def fit_formula_glm_bundle(model_key: str, formula: str, train_df: pd.DataFrame, holdout_df: pd.DataFrame) -> dict:
    features = extract_formula_features(formula)
    required_cols = features + [
        ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL, "Policies_Exposed", "Amount_Exposed"
    ]

    train_model_df = prepare_model_frame(train_df, required_cols)
    holdout_model_df = prepare_model_frame(holdout_df, required_cols)

    train_cells = aggregate_cells(train_model_df, features)

    result = smf.glm(
        formula=formula,
        data=train_cells,
        family=sm.families.Poisson(),
        offset=np.log(train_cells[EXPECTED_CNT_COL]),
    ).fit()

    bundle = {
        "model_key": model_key,
        "model_type": "statsmodels_glm",
        "label": model_key,
        "formula": formula,
        "features": features,
        "result": result,
    }

    scored_train = score_with_bundle(bundle, train_model_df)
    scored_holdout = score_with_bundle(bundle, holdout_model_df)

    metrics = pd.DataFrame([
        evaluate_predictions(scored_train[ACTUAL_CNT_COL], scored_train[EXPECTED_CNT_COL], scored_train["Predicted_Deaths"], "Train"),
        evaluate_predictions(scored_holdout[ACTUAL_CNT_COL], scored_holdout[EXPECTED_CNT_COL], scored_holdout["Predicted_Deaths"], "Holdout"),
    ])

    bundle["metrics"] = metrics
    bundle["train_cells"] = train_cells
    return bundle


def fit_poisson_regressor_bundle(
    model_key: str,
    train_df: pd.DataFrame,
    holdout_df: pd.DataFrame,
    features: list[str],
    alpha: float = 0.001,
) -> dict:
    require_sklearn()

    needed = features + [ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL, "Policies_Exposed", "Amount_Exposed"]
    train_model_df = prepare_model_frame(train_df, needed)
    holdout_model_df = prepare_model_frame(holdout_df, needed)

    train_cells = aggregate_cells(train_model_df, features)

    category_levels = build_category_levels(train_cells, features)
    X_train = encode_features(train_cells, features, category_levels=category_levels)

    rate_train = safe_divide(train_cells[ACTUAL_CNT_COL], train_cells[EXPECTED_CNT_COL])
    w_train = np.clip(train_cells[EXPECTED_CNT_COL].to_numpy(dtype=float), 1e-12, None)

    est = PoissonRegressor(alpha=alpha, max_iter=1000)
    est.fit(X_train, rate_train, sample_weight=w_train)

    bundle = {
        "model_key": model_key,
        "model_type": "poisson_regressor",
        "label": model_key,
        "features": features,
        "result": est,
        "category_levels": category_levels,
        "design_columns": X_train.columns.tolist(),
    }

    scored_train = score_with_bundle(bundle, train_model_df)
    scored_holdout = score_with_bundle(bundle, holdout_model_df)

    metrics = pd.DataFrame([
        evaluate_predictions(scored_train[ACTUAL_CNT_COL], scored_train[EXPECTED_CNT_COL], scored_train["Predicted_Deaths"], "Train"),
        evaluate_predictions(scored_holdout[ACTUAL_CNT_COL], scored_holdout[EXPECTED_CNT_COL], scored_holdout["Predicted_Deaths"], "Holdout"),
    ])

    bundle["metrics"] = metrics
    bundle["train_cells"] = train_cells
    return bundle


def fit_log_ratio_bundle(
    model_key: str,
    estimator,
    train_df: pd.DataFrame,
    holdout_df: pd.DataFrame,
    features: list[str],
    smooth: float = 0.25,
) -> dict:
    require_sklearn()

    needed = features + [ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL, "Policies_Exposed", "Amount_Exposed"]
    train_model_df = prepare_model_frame(train_df, needed)
    holdout_model_df = prepare_model_frame(holdout_df, needed)

    train_cells = aggregate_cells(train_model_df, features)

    category_levels = build_category_levels(train_cells, features)
    X_train = encode_features(train_cells, features, category_levels=category_levels)

    y_train = np.log((train_cells[ACTUAL_CNT_COL] + smooth) / (train_cells[EXPECTED_CNT_COL] + smooth))
    w_train = np.clip(train_cells[EXPECTED_CNT_COL].to_numpy(dtype=float), 1e-12, None)

    estimator.fit(X_train, y_train, sample_weight=w_train)

    bundle = {
        "model_key": model_key,
        "model_type": "log_ratio_model",
        "label": model_key,
        "features": features,
        "result": estimator,
        "category_levels": category_levels,
        "design_columns": X_train.columns.tolist(),
        "smooth": smooth,
    }

    scored_train = score_with_bundle(bundle, train_model_df)
    scored_holdout = score_with_bundle(bundle, holdout_model_df)

    metrics = pd.DataFrame([
        evaluate_predictions(scored_train[ACTUAL_CNT_COL], scored_train[EXPECTED_CNT_COL], scored_train["Predicted_Deaths"], "Train"),
        evaluate_predictions(scored_holdout[ACTUAL_CNT_COL], scored_holdout[EXPECTED_CNT_COL], scored_holdout["Predicted_Deaths"], "Holdout"),
    ])

    bundle["metrics"] = metrics
    bundle["train_cells"] = train_cells
    return bundle


def fit_log_rate_bundle(*args, **kwargs):
    return fit_log_ratio_bundle(*args, **kwargs)


def bundle_comparison_row(bundle: dict) -> dict:
    metrics = bundle["metrics"].set_index("Dataset")
    return {
        "Model_Key": bundle["model_key"],
        "Model_Type": bundle["model_type"],
        "Feature_Count": len(bundle["features"]),
        "Train_Deviance": float(metrics.loc["Train", "Poisson_Deviance"]),
        "Holdout_Deviance": float(metrics.loc["Holdout", "Poisson_Deviance"]),
        "Train_A/P": float(metrics.loc["Train", "Actual_to_Predicted"]),
        "Holdout_A/P": float(metrics.loc["Holdout", "Actual_to_Predicted"]),
        "Train_Weighted_AE_Error": float(metrics.loc["Train", "Weighted_AE_Error"]),
        "Holdout_Weighted_AE_Error": float(metrics.loc["Holdout", "Weighted_AE_Error"]),
    }


def add_validation_bands(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    out["Issue_Age_Band"] = pd.cut(
        out["Issue_Age"],
        bins=[-0.1, 0, 5, 10, 15, 17],
        labels=["0", "1-5", "6-10", "11-15", "16-17"],
        include_lowest=True,
        right=True,
    )

    attained_min = int(np.floor(out["Attained_Age"].min())) if len(out) else 0
    attained_bins = list(range(max(0, attained_min // 5 * 5), 101, 5))
    if attained_bins[-1] < 100:
        attained_bins.append(100)

    out["Attained_Age_Band"] = pd.cut(
        out["Attained_Age"],
        bins=attained_bins,
        include_lowest=True,
        right=False,
    )

    out["Duration_Band"] = pd.cut(
        np.where(out["Duration"] >= 26, 26, out["Duration"]),
        bins=[-0.1, 1, 5, 10, 15, 20, 26],
        labels=["0-1", "2-5", "6-10", "11-15", "16-20", "21-26+"],
        include_lowest=True,
        right=True,
    )
    return out


def calibration_table(df: pd.DataFrame, group_cols: str | list[str], pred_col: str = "Predicted_Deaths") -> pd.DataFrame:
    if isinstance(group_cols, str):
        group_cols = [group_cols]

    out = (
        df.groupby(group_cols, dropna=False)[
            [ACTUAL_CNT_COL, EXPECTED_CNT_COL, ACTUAL_AMT_COL, EXPECTED_AMT_COL, "Policies_Exposed", pred_col]
        ]
        .sum()
        .reset_index()
    )

    out["Actual_to_Predicted"] = safe_divide(out[ACTUAL_CNT_COL], out[pred_col])
    out["Actual_to_VBT"] = safe_divide(out[ACTUAL_CNT_COL], out[EXPECTED_CNT_COL])
    out["Predicted_to_VBT"] = safe_divide(out[pred_col], out[EXPECTED_CNT_COL])
    out["Amount_AE_to_VBT"] = safe_divide(out[ACTUAL_AMT_COL], out[EXPECTED_AMT_COL])
    out["Actual_Rate_per_1000"] = safe_divide(out[ACTUAL_CNT_COL] * 1000.0, out["Policies_Exposed"])
    out["Expected_Rate_per_1000"] = safe_divide(out[EXPECTED_CNT_COL] * 1000.0, out["Policies_Exposed"])
    out["Predicted_Rate_per_1000"] = safe_divide(out[pred_col] * 1000.0, out["Policies_Exposed"])
    out["Gap_from_1_on_A/P"] = (out["Actual_to_Predicted"] - 1.0).abs()

    out["Flag"] = np.select(
        [
            out[EXPECTED_CNT_COL] < MIN_EXPECTED_FOR_REVIEW,
            out["Gap_from_1_on_A/P"] > SEGMENT_TOLERANCE,
        ],
        [
            "Thin",
            "Review",
        ],
        default="OK",
    )

    out["Flag_Order"] = out["Flag"].map({"Review": 2, "Thin": 1, "OK": 0}).fillna(0)
    return out


def residual_heatmap(df: pd.DataFrame, row_col: str, col_col: str, value_col: str = "Actual_to_Predicted"):
    heat = calibration_table(df, [row_col, col_col]).pivot(index=row_col, columns=col_col, values=value_col)

    plt.figure(figsize=(10, 6))
    plt.imshow(heat.values, aspect="auto")
    plt.colorbar(label=value_col)
    plt.xticks(range(len(heat.columns)), [str(c) for c in heat.columns], rotation=45, ha="right")
    plt.yticks(range(len(heat.index)), [str(i) for i in heat.index])
    plt.title(f"{value_col}: {row_col} × {col_col}")
    plt.tight_layout()
    plt.show()


def build_sheet_style_table(
    df: pd.DataFrame,
    value_col: str,
    age_ind_value: str,
    sex_value: str,
    select_period: int = 25,
    ult_duration: int = 26,
    duration_cols: list[int] | None = None,
    rate_per: float = 1000.0,
    issue_age_min: int = 0,
    issue_age_max: int = 17,
) -> pd.DataFrame:
    if duration_cols is None:
        duration_cols = list(range(1, 26))

    work = df.copy()
    work["Age_Ind"] = work["Age_Ind"].astype("string")
    work["Sex"] = work["Sex"].astype("string")

    work = work[
        (work["Age_Ind"] == str(age_ind_value)) &
        (work["Sex"] == str(sex_value)) &
        (work["Issue_Age"].between(issue_age_min, issue_age_max, inclusive="both"))
    ].copy()

    rows = []
    for issue_age in range(issue_age_min, issue_age_max + 1):
        row = {"Iss. Age": issue_age}

        for dur in duration_cols:
            sub = work[(work["Issue_Age"] == issue_age) & (work["Duration"] == dur)].copy()
            row[str(dur)] = safe_divide(rate_per * sub[value_col].sum(), sub["Policies_Exposed"].sum())

        ult_sub = work[(work["Issue_Age"] == issue_age)].copy()

        preferred_ult = ult_sub[(ult_sub["Duration"] >= ult_duration)].copy()

        if "Slct_Ult_Ind" in ult_sub.columns and len(preferred_ult) == 0:
            preferred_ult = ult_sub[
                ult_sub["Slct_Ult_Ind"].astype("string").str.upper().isin(["U", "ULT", "ULTIMATE"])
            ].copy()

        if len(preferred_ult) > 0 and "Attained_Age" in preferred_ult.columns:
            exact_age = preferred_ult[preferred_ult["Attained_Age"] == issue_age + select_period].copy()
            if len(exact_age) > 0:
                preferred_ult = exact_age

        row["Ult."] = safe_divide(rate_per * preferred_ult[value_col].sum(), preferred_ult["Policies_Exposed"].sum())
        row["Att. Age"] = issue_age + select_period
        rows.append(row)

    col_order = ["Iss. Age"] + [str(d) for d in duration_cols] + ["Ult.", "Att. Age"]
    return pd.DataFrame(rows)[col_order]


def to_long_sheet_table(sheet_df: pd.DataFrame) -> pd.DataFrame:
    long_df = sheet_df.melt(id_vars=["Iss. Age", "Att. Age"], var_name="Duration_Col", value_name="Rate")
    return long_df


def read_reference_sheet(path: Path, sheet_name: str, max_issue_age: int = 17) -> pd.DataFrame:
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None)

    header_idx = raw.index[raw.iloc[:, 0].astype("string").str.contains("Iss. Age", na=False)]
    if len(header_idx) == 0:
        raise ValueError("Could not find the 'Iss. Age' header row in the benchmark sheet.")
    header_idx = int(header_idx[0])

    data = raw.iloc[header_idx + 1:].copy()
    data["IssueAgeNumeric"] = pd.to_numeric(data.iloc[:, 0], errors="coerce")
    data = data[data["IssueAgeNumeric"].between(0, max_issue_age, inclusive="both")].copy()

    out = data.iloc[:, :28].copy()
    out.columns = ["Iss. Age"] + [str(i) for i in range(1, 26)] + ["Ult.", "Att. Age"]
    out["Iss. Age"] = pd.to_numeric(out["Iss. Age"], errors="coerce")
    out["Att. Age"] = pd.to_numeric(out["Att. Age"], errors="coerce")

    value_cols = [c for c in out.columns if c not in ["Iss. Age", "Att. Age"]]
    for col in value_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    return out.reset_index(drop=True)


def compare_to_reference(modeled_sheet: pd.DataFrame, reference_sheet: pd.DataFrame) -> pd.DataFrame:
    modeled_long = to_long_sheet_table(modeled_sheet).rename(columns={"Rate": "Modeled_Rate"})
    reference_long = to_long_sheet_table(reference_sheet).rename(columns={"Rate": "Reference_Rate"})

    out = modeled_long.merge(
        reference_long,
        on=["Iss. Age", "Att. Age", "Duration_Col"],
        how="inner",
    )
    out["Difference"] = out["Modeled_Rate"] - out["Reference_Rate"]
    out["Absolute_Difference"] = out["Difference"].abs()
    return out.sort_values(["Absolute_Difference", "Iss. Age"], ascending=[False, True]).reset_index(drop=True)


In [16]:
# ============================================================
# LOAD DATA, APPLY SCOPE, AND REVIEW OVERVIEW
# ============================================================

raw_df = read_parquet_data(DATA_PATH, columns=BASE_COLUMNS)
scoped_df, scope_log = apply_scope_filters(raw_df)

if HOLDOUT_YEARS is None:
    screening_df = scoped_df.copy()
else:
    screening_df = scoped_df[~scoped_df["Observation_Year"].isin(HOLDOUT_YEARS)].copy()

print("Scope log")
display(scope_log)

print("\nOverall scoped metrics")
display(overall_metrics(scoped_df))

print("\nScreening base metrics (used in Session 2)")
display(overall_metrics(screening_df))

# export_csv(scope_log, "session1_scope_log.csv")
# export_csv(overall_metrics(scoped_df), "session1_overall_metrics.csv")
# export_csv(overall_metrics(screening_df), "session1_screening_metrics.csv")

Scope log


,step,rows
0,raw_input,3598960
1,Issue_Age between 0 and 17,3598960
2,Attained_Age >= 1,3592746
3,Attained_Age <= 94,3586281
4,Smoker_Status in ['U'],1680838
5,Insurance_Plan not in ['Other'],1563079
6,SOA_Post_Lvl_Ind not in ['PLT'],1533018
7,ExpDth_VBT2015_Cnt > 0,1494745
8,Applied 1M face cap rule,1494745



Overall scoped metrics


,Metric,Value
0,Rows,"1,494,745.000000"
1,Policies Exposed,"66,851,871.164963"
2,Amount Exposed,"2,483,050,339,115.433594"
3,Death Count,"333,551.000000"
4,Expected Death Count,"327,860.433647"
5,A/E Count,1.017357
6,Death Claim Amount,"2,815,301,540.000000"
7,Expected Death Amount,"2,612,603,820.369946"
8,A/E Amount,1.077585
9,Actual Rate per 1000,4.989404



Screening base metrics (used in Session 2)


,Metric,Value
0,Rows,"927,694.000000"
1,Policies Exposed,"46,930,429.480793"
2,Amount Exposed,"1,655,995,217,232.724121"
3,Death Count,"243,771.000000"
4,Expected Death Count,"236,040.419122"
5,A/E Count,1.032751
6,Death Claim Amount,"1,995,181,102.000000"
7,Expected Death Amount,"1,843,762,442.677902"
8,A/E Amount,1.082125
9,Actual Rate per 1000,5.194306


In [17]:

# Quick structure and sample rows
print("Columns loaded:")
print(list(scoped_df.columns))

display(scoped_df.head(8))


Columns loaded:
['Observation_Year', 'Issue_Age', 'Duration', 'Issue_Year', 'Attained_Age', 'Amount_Exposed', 'Policies_Exposed', 'Death_Claim_Amount', 'Death_Count', 'ExpDth_VBT2015_Cnt', 'ExpDth_VBT2015_Amt', 'Age_Ind', 'Sex', 'Smoker_Status', 'Insurance_Plan', 'Face_Amount_Band', 'SOA_Antp_Lvl_TP', 'SOA_Guar_Lvl_TP', 'SOA_Post_Lvl_Ind', 'Slct_Ult_Ind', 'Preferred_Indicator', 'Number_of_Pfd_Classes', 'Preferred_Class']


,Observation_Year,Issue_Age,Duration,Issue_Year,Attained_Age,Amount_Exposed,Policies_Exposed,Death_Claim_Amount,Death_Count,ExpDth_VBT2015_Cnt,ExpDth_VBT2015_Amt,Age_Ind,Sex,Smoker_Status,Insurance_Plan,Face_Amount_Band,SOA_Antp_Lvl_TP,SOA_Guar_Lvl_TP,SOA_Post_Lvl_Ind,Slct_Ult_Ind,Preferred_Indicator,Number_of_Pfd_Classes,Preferred_Class
0,2012,0,2,2011,1,"20,506.857422",3.934248,0,0,0.000472,2.460823,ALB,F,U,Perm,"01: 0 - 9,999",N/A (Not Term),N/A (Not Term),N/A,S,0,NA,NA
1,2012,0,2,2010,1,"11,038,165.000000",425.346375,0,0,0.051042,"1,324.579834",ALB,F,U,Perm,"03: 25,000 - 49,999",N/A (Not Term),N/A (Not Term),N/A,S,0,NA,NA
2,2012,0,2,2010,1,"15,043,168.000000",294.245728,0,0,0.035309,"1,805.180176",ALB,F,U,Perm,"04: 50,000 - 99,999",N/A (Not Term),N/A (Not Term),N/A,S,0,NA,NA
3,2012,0,2,2010,1,"7,187,517.500000",68.218666,0,0,0.008186,862.502075,ALB,F,U,Perm,"05: 100,000 - 249,999",N/A (Not Term),N/A (Not Term),N/A,S,0,NA,NA
4,2012,0,3,2009,2,"3,703,845.000000",277.395508,0,0,0.022192,296.307587,ALB,F,U,Perm,"02: 10,000 - 24,999",N/A (Not Term),N/A (Not Term),N/A,S,0,NA,NA
5,2012,0,3,2010,2,"667,847.250000",2.561780,0,0,0.000205,53.427780,ALB,F,U,Perm,"06: 250,000 - 499,999",N/A (Not Term),N/A (Not Term),N/A,S,0,NA,NA
6,2012,0,4,2008,3,"11,791,422.000000",43.238670,0,0,0.003027,825.399536,ALB,F,U,Perm,"06: 250,000 - 499,999",N/A (Not Term),N/A (Not Term),N/A,S,0,NA,NA
7,2012,0,4,2009,3,"903,451.312500",3.457536,0,0,0.000242,63.241592,ALB,F,U,Perm,"06: 250,000 - 499,999",N/A (Not Term),N/A (Not Term),N/A,S,0,NA,NA


In [18]:
# One-way A/E review for the most decision-relevant dimensions
for dim in ["Age_Ind", "Sex", "Insurance_Plan", "Face_Amount_Band", "Observation_Year"]:
    print(f"\n=== {dim} ===")
    display(ae_summary(scoped_df, dim))


=== Age_Ind ===


,Age_Ind,Death_Count,ExpDth_VBT2015_Cnt,Death_Claim_Amount,ExpDth_VBT2015_Amt,Policies_Exposed,Amount_Exposed,A/E_Count,A/E_Amount,Actual_Rate_per_1000,Expected_Rate_per_1000
0,ALB,86120,"68,411.633923",1021675965,"799,799,336.832713","34,278,694.595144","1,084,419,306,833.075562",1.258850,1.277415,2.512348,1.995748
1,ANB,247431,"259,448.799723",1793625575,"1,812,804,483.537234","32,573,176.569818","1,398,631,032,282.357666",0.953679,0.989420,7.596158,7.965106



=== Sex ===


,Sex,Death_Count,ExpDth_VBT2015_Cnt,Death_Claim_Amount,ExpDth_VBT2015_Amt,Policies_Exposed,Amount_Exposed,A/E_Count,A/E_Amount,Actual_Rate_per_1000,Expected_Rate_per_1000
0,F,85668,"87,033.888610",662914354,"636,980,783.920792","29,406,841.947654","1,184,635,744,290.353027",0.984306,1.040713,2.913200,2.959648
1,M,247883,"240,826.545036",2152387186,"1,975,623,036.449154","37,445,029.217308","1,298,414,594,825.080078",1.029301,1.089473,6.619917,6.431469



=== Insurance_Plan ===


,Insurance_Plan,Death_Count,ExpDth_VBT2015_Cnt,Death_Claim_Amount,ExpDth_VBT2015_Amt,Policies_Exposed,Amount_Exposed,A/E_Count,A/E_Amount,Actual_Rate_per_1000,Expected_Rate_per_1000
0,Perm,321941,"315,792.257438",2322346625,"2,214,100,735.437118","54,774,073.971190","1,678,985,637,033.704102",1.019471,1.048889,5.877616,5.765360
1,Term,4391,"6,875.933699",41445363,"39,186,870.031659","2,659,089.317488","55,607,106,111.774986",0.638604,1.057634,1.651317,2.585823
2,UL,5942,"4,134.139728",321359340,"237,961,942.574435","6,965,359.491376","416,135,573,089.046936",1.437300,1.350465,0.853079,0.593529
3,ULSG,278,243.549776,24228933,"28,210,647.813859","903,912.119715","119,043,086,238.299759",1.141450,0.858858,0.307552,0.269440
4,VL,795,629.082437,74247643,"63,289,150.119932","1,024,407.033695","121,541,244,101.005814",1.263745,1.173150,0.776059,0.614094
5,VLSG,204,185.470569,31673636,"29,854,474.392943","525,029.231498","91,737,692,541.601715",1.099905,1.060934,0.388550,0.353258



=== Face_Amount_Band ===


,Face_Amount_Band,Death_Count,ExpDth_VBT2015_Cnt,Death_Claim_Amount,ExpDth_VBT2015_Amt,Policies_Exposed,Amount_Exposed,A/E_Count,A/E_Amount,Actual_Rate_per_1000,Expected_Rate_per_1000
0,"01: 0 - 9,999",270290,"274,988.203098",809766460,"798,300,103.167178","24,635,681.651337","97,414,029,794.255875",0.982915,1.014363,10.971485,11.162192
1,"02: 10,000 - 24,999",40380,"33,231.521230",608437006,"515,648,380.601652","13,758,174.128474","209,023,679,654.567505",1.215111,1.179946,2.934982,2.415402
2,"03: 25,000 - 49,999",13112,"11,013.057534",432506877,"374,959,322.834609","13,515,016.748943","397,372,198,783.362488",1.190587,1.153477,0.970180,0.814876
3,"04: 50,000 - 99,999",6587,"5,443.425812",406381546,"345,888,774.880593","8,276,023.277180","476,639,819,066.810486",1.210084,1.174891,0.795914,0.657734
4,"05: 100,000 - 249,999",2683,"2,599.194565",347990161,"339,168,471.085764","5,033,675.417800","633,184,996,309.440308",1.032243,1.026010,0.533010,0.516361
5,"06: 250,000 - 499,999",368,430.683291,114179363,"129,984,790.895418","1,153,316.315770","336,747,016,322.013428",0.854456,0.878406,0.319080,0.373430
6,"07: 500,000 - 999,999",89,112.183956,54040127,"66,489,816.608244","349,028.852376","201,713,826,101.120728",0.793340,0.812758,0.254993,0.321417
7,"08: 1,000,000+",42,42.164160,42000000,"42,164,160.296488","130,954.773084","130,954,773,083.862381",0.996107,0.996107,0.320721,0.321975



=== Observation_Year ===


,Observation_Year,Death_Count,ExpDth_VBT2015_Cnt,Death_Claim_Amount,ExpDth_VBT2015_Amt,Policies_Exposed,Amount_Exposed,A/E_Count,A/E_Amount,Actual_Rate_per_1000,Expected_Rate_per_1000
0,2012,40388,"38,994.414445",258853169,"239,071,530.323833","6,537,836.129823","192,339,960,478.046539",1.035738,1.082744,6.177579,5.964422
1,2013,38937,"37,732.063441",295147011,"280,901,322.011556","8,165,995.769880","268,408,840,714.527740",1.031934,1.050714,4.768188,4.620632
2,2014,40716,"38,922.054459",328727659,"300,120,184.713004","8,071,923.269631","275,558,251,998.135986",1.046091,1.095320,5.044151,4.821906
3,2015,41156,"39,154.441337",352339438,"320,864,569.169383","8,079,322.221069","293,244,272,128.005005",1.051120,1.098094,5.093992,4.846253
4,2016,40463,"39,977.039576",369969338,"342,087,810.722955","8,084,550.454448","309,765,624,958.495361",1.012156,1.081504,5.004978,4.944869
5,2017,42111,"41,260.405865",390144487,"360,717,025.737170","7,990,801.635941","316,678,266,955.513672",1.020615,1.081580,5.269934,5.163488
6,2018,45269,"45,503.804212",407078025,"375,192,004.778064","10,029,021.002603","407,247,052,906.454895",0.994840,1.084986,4.513800,4.537213
7,2019,44511,"46,316.210312",413042413,"393,649,372.913980","9,892,420.681567","419,808,068,976.254089",0.961024,1.049265,4.499505,4.681990


We notice:
1. Female:male and ALB:ANB mixes are each roughly 1:3.
2. Permanent and term business show materially different mortality patterns, possibly reflecting higher term participation at larger face amounts.
3. Mortality generally trends downward as face amount increases.
4. No clear MI trend is observed over time.





---

# Session 2 · Model screening / feature selection

This section gives you several simple ways to judge feature strength before you lock in the model structure.

In [20]:
# ============================================================
# SESSION 2A — UNIVARIATE GLM SCREEN
# ============================================================

screen_input = screening_df[SCREEN_FEATURES + [ACTUAL_CNT_COL, EXPECTED_CNT_COL]].copy()
screen_results = run_univariate_screen(screen_input, SCREEN_FEATURES)

display(screen_results)
export_csv(screen_results, "session2_univariate_screen.csv")

C:\Users\sober\PycharmProjects\Juvenile\.venv\Lib\site-packages\statsmodels\regression\_tools.py:121: RuntimeWarning: divide by zero encountered in scalar divide
  scale = np.dot(wresid, wresid) / df_resid
C:\Users\sober\PycharmProjects\Juvenile\.venv\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
C:\Users\sober\PycharmProjects\Juvenile\.venv\Lib\site-packages\statsmodels\regression\_tools.py:121: RuntimeWarning: divide by zero encountered in scalar divide
  scale = np.dot(wresid, wresid) / df_resid
C:\Users\sober\PycharmProjects\Juvenile\.venv\Lib\site-packages\statsmodels\genmod\generalized_linear_model.py:1342: PerfectSeparationWarning: Perfect separation or prediction detected, parameter may not be identified
  warnings.warn(msg, category=PerfectSeparationWarning)
C:\Users\sober\PycharmProjects\Juvenile\

,Feature,Levels_or_Unique,Base_Deviance,Model_Deviance,Deviance_Reduction,Deviance_Reduction_%,AIC,Weighted_AE_Error
0,Insurance_Plan,6,370.176356,-0.000000,370.176356,100.000000,66.381414,0.000000
1,Face_Amount_Band,8,951.187154,0.000000,951.187154,100.000000,91.605880,0.000000
2,Age_Ind,2,"3,653.282799",0.000000,"3,653.282799",100.000000,30.801330,0.000000
3,Observation_Year,6,42.191620,0.000000,42.191620,100.000000,86.698984,0.000000
4,Slct_Ult_Ind,2,348.469569,0.000000,348.469569,100.000000,29.120893,0.000000
5,SOA_Post_Lvl_Ind,4,40.042081,0.000000,40.042081,100.000000,40.503001,0.000000
6,Sex,2,64.400941,0.000000,64.400941,100.000000,30.817767,0.000000
7,SOA_Guar_Lvl_TP,9,41.285143,0.000000,41.285143,100.000000,60.955143,0.000000
8,SOA_Antp_Lvl_TP,9,41.285143,0.000000,41.285143,100.000000,60.955143,0.000000
9,Duration,95,"11,163.368566",373.441302,"10,789.927263",96.654762,"1,226.913989",0.026675


Saved: script\model_outputs\session2_univariate_screen.csv


In [22]:
# ============================================================
# SESSION 2B — GBM FEATURE IMPORTANCE
# ============================================================

require_sklearn()

gbm_screen_results = run_tree_feature_importance(
    screening_df,
    SCREEN_FEATURES,
    estimator=GradientBoostingRegressor(random_state=SCREEN_RANDOM_STATE),
    smooth=SCREEN_TARGET_SMOOTH,
    random_state=SCREEN_RANDOM_STATE,
)

display(gbm_screen_results)
export_csv(gbm_screen_results, "session2_gbm_feature_importance.csv")

,Feature,Importance,Rank
0,Duration,0.593443,1
1,Observation_Year,0.177382,2
2,Attained_Age,0.092383,3
3,Issue_Age,0.049348,4
4,Face_Amount_Band,0.037059,5
5,Sex,0.028891,6
6,Age_Ind,0.021173,7
7,Insurance_Plan,0.000209,8
8,SOA_Post_Lvl_Ind,0.000068,9
9,SOA_Antp_Lvl_TP,0.000043,10


Saved: script\model_outputs\session2_gbm_feature_importance.csv


In [23]:
# ============================================================
# SESSION 2C — EXTRATREES FEATURE IMPORTANCE
# ============================================================

require_sklearn()

extratrees_screen_results = run_tree_feature_importance(
    screening_df,
    SCREEN_FEATURES,
    estimator=ExtraTreesRegressor(
        n_estimators=300,
        random_state=SCREEN_RANDOM_STATE,
        n_jobs=-1,
        min_samples_leaf=5,
    ),
    smooth=SCREEN_TARGET_SMOOTH,
    random_state=SCREEN_RANDOM_STATE,
)

display(extratrees_screen_results)
export_csv(extratrees_screen_results, "session2_extratrees_feature_importance.csv")

,Feature,Importance,Rank
0,Duration,0.414480,1
1,Observation_Year,0.182007,2
2,Attained_Age,0.168354,3
3,Issue_Age,0.148145,4
4,Face_Amount_Band,0.033131,5
5,Sex,0.030436,6
6,Age_Ind,0.018398,7
7,Insurance_Plan,0.002277,8
8,Slct_Ult_Ind,0.002158,9
9,SOA_Guar_Lvl_TP,0.000211,10


Saved: script\model_outputs\session2_extratrees_feature_importance.csv


In [24]:

# ============================================================
# SESSION 2D — COMBINED SCREENING SCOREBOARD
# ============================================================

available_univariate = screen_results if "screen_results" in globals() else None
available_gbm = gbm_screen_results if "gbm_screen_results" in globals() else None
available_extra = extratrees_screen_results if "extratrees_screen_results" in globals() else None

screen_scoreboard = combine_screen_ranks(
    univariate_df=available_univariate,
    gbm_df=available_gbm,
    extra_df=available_extra,
)

display(screen_scoreboard)
export_csv(screen_scoreboard, "session2_combined_screen_scoreboard.csv")


,Feature,Deviance_Reduction_%,Univariate_Rank,GBM_Importance,GBM_Rank,ExtraTrees_Importance,ExtraTrees_Rank,Average_Rank
0,Observation_Year,100.000000,4.000000,0.177382,2.000000,0.182007,2.000000,2.666667
1,Duration,96.654762,9.000000,0.593443,1.000000,0.414480,1.000000,3.666667
2,Face_Amount_Band,100.000000,2.000000,0.037059,5.000000,0.033131,5.000000,4.000000
3,Attained_Age,89.550211,10.000000,0.092383,3.000000,0.168354,3.000000,5.333333
4,Insurance_Plan,100.000000,1.000000,0.000209,8.000000,0.002277,8.000000,5.666667
5,Age_Ind,100.000000,3.000000,0.021173,7.000000,0.018398,7.000000,5.666667
6,Sex,100.000000,7.000000,0.028891,6.000000,0.030436,6.000000,6.333333
7,Issue_Age,81.394968,11.000000,0.049348,4.000000,0.148145,4.000000,6.333333
8,Slct_Ult_Ind,100.000000,5.000000,0.000000,11.000000,0.002158,9.000000,8.333333
9,SOA_Post_Lvl_Ind,100.000000,6.000000,0.000068,9.000000,0.000201,12.000000,9.000000


Saved: script\model_outputs\session2_combined_screen_scoreboard.csv


We conclude:

1. The current mortality structure (Attained age by Issue age, by sex and age ind) is reasonable.
2. Select or Ultimate, Post level, Antp, Guar, preferred class and indicator are not considered in the model.
3. Need closer examination of Face amount band and Insurance plan.

---

# Session 3 · Model fitting and model comparison

This section uses multiple explicit model cells so the results stay easy to inspect and compare.

In [29]:
# ============================================================
# BUILD TRAIN / HOLDOUT SETS
# ============================================================

if HOLDOUT_YEARS is None:
    train_raw = scoped_df.copy()
    holdout_raw = scoped_df.copy()
    print("No holdout years supplied: train and evaluation both use full scoped data.")
else:
    train_raw = scoped_df[~scoped_df["Observation_Year"].isin(HOLDOUT_YEARS)].copy()
    holdout_raw = scoped_df[scoped_df["Observation_Year"].isin(HOLDOUT_YEARS)].copy()
    print(f"Train years  : {sorted(train_raw['Observation_Year'].dropna().unique().tolist())}")
    print(f"Holdout years: {sorted(holdout_raw['Observation_Year'].dropna().unique().tolist())}")

true_holdout_available = (HOLDOUT_YEARS is not None) and (len(holdout_raw) > 0)

print(f"Train rows  : {len(train_raw):,}")
print(f"Holdout rows: {len(holdout_raw):,}")

MODEL_BUNDLES = {}
MODEL_ROWS = []

def save_model_bundle(bundle: dict):
    MODEL_BUNDLES[bundle["model_key"]] = bundle
    MODEL_ROWS.append(bundle_comparison_row(bundle))
    print(f"Saved model: {bundle['model_key']}")
    display(bundle["metrics"])

No holdout years supplied: train and evaluation both use full scoped data.
Train rows  : 1,494,745
Holdout rows: 1,494,745


In [30]:
# ============================================================
# MODEL 1 — POISSON GLM CORE
# ============================================================

try:
    bundle_poisson_glm_core = fit_formula_glm_bundle(
        model_key="poisson_glm_core",
        formula=GLM_FORMULA_CORE,
        train_df=train_raw,
        holdout_df=holdout_raw,
    )
    save_model_bundle(bundle_poisson_glm_core)
except Exception as exc:
    print(f"Model failed: poisson_glm_core -> {exc}")


Saved model: poisson_glm_core


,Dataset,Rows,Actual_Deaths,Expected_VBT,Predicted_Deaths,Actual_to_Predicted,Actual_to_VBT,Poisson_Deviance,Weighted_AE_Error
0,Train,1494745,"333,551.000000","327,860.433647","333,551.000000",1.000000,1.017357,"279,678.905112",0.521286
1,Holdout,1494745,"333,551.000000","327,860.433647","333,551.000000",1.000000,1.017357,"279,678.905112",0.521286


In [31]:
# ============================================================
# MODEL 2 — POISSON GLM ATTAINED-AGE VERSION
# ============================================================

try:
    bundle_poisson_glm_attained = fit_formula_glm_bundle(
        model_key="poisson_glm_attained",
        formula=GLM_FORMULA_ATTAINED,
        train_df=train_raw,
        holdout_df=holdout_raw,
    )
    save_model_bundle(bundle_poisson_glm_attained)
except Exception as exc:
    print(f"Model failed: poisson_glm_attained -> {exc}")


Saved model: poisson_glm_attained


,Dataset,Rows,Actual_Deaths,Expected_VBT,Predicted_Deaths,Actual_to_Predicted,Actual_to_VBT,Poisson_Deviance,Weighted_AE_Error
0,Train,1494745,"333,551.000000","327,860.433647","333,551.000000",1.000000,1.017357,"278,118.614037",0.517760
1,Holdout,1494745,"333,551.000000","327,860.433647","333,551.000000",1.000000,1.017357,"278,118.614037",0.517760


In [32]:
# ============================================================
# MODEL 3 — SKLEARN POISSON REGRESSOR
# ============================================================

try:
    bundle_poisson_reg = fit_poisson_regressor_bundle(
        model_key="poisson_regressor",
        train_df=train_raw,
        holdout_df=holdout_raw,
        features=ML_FEATURES,
        alpha=0.001,
    )
    save_model_bundle(bundle_poisson_reg)
except Exception as exc:
    print(f"Model failed: poisson_regressor -> {exc}")


Saved model: poisson_regressor


,Dataset,Rows,Actual_Deaths,Expected_VBT,Predicted_Deaths,Actual_to_Predicted,Actual_to_VBT,Poisson_Deviance,Weighted_AE_Error
0,Train,1494745,"333,551.000000","327,860.433647","333,534.231715",1.000050,1.017357,"280,417.070477",0.522657
1,Holdout,1494745,"333,551.000000","327,860.433647","333,534.231715",1.000050,1.017357,"280,417.070477",0.522657


In [33]:
# ============================================================
# MODEL 4 — RANDOM FOREST MORTALITY RATIO MODEL
# ============================================================

try:
    bundle_random_forest = fit_log_ratio_bundle(
        model_key="random_forest_log_ratio",
        estimator=RandomForestRegressor(
            n_estimators=400,
            min_samples_leaf=5,
            random_state=MODEL_RANDOM_STATE,
            n_jobs=-1,
        ),
        train_df=train_raw,
        holdout_df=holdout_raw,
        features=ML_FEATURES,
        smooth=MODEL_TARGET_SMOOTH,
    )
    save_model_bundle(bundle_random_forest)
except Exception as exc:
    print(f"Model failed: random_forest_log_ratio -> {exc}")

Saved model: random_forest_log_ratio


,Dataset,Rows,Actual_Deaths,Expected_VBT,Predicted_Deaths,Actual_to_Predicted,Actual_to_VBT,Poisson_Deviance,Weighted_AE_Error
0,Train,1494745,"333,551.000000","327,860.433647","322,374.662411",1.034669,1.017357,"261,046.588670",0.480340
1,Holdout,1494745,"333,551.000000","327,860.433647","322,374.662411",1.034669,1.017357,"261,046.588670",0.480340


In [34]:
# ============================================================
# MODEL 5 — GRADIENT BOOSTING MORTALITY RATIO MODEL
# ============================================================

try:
    bundle_gbm = fit_log_ratio_bundle(
        model_key="gradient_boosting_log_ratio",
        estimator=GradientBoostingRegressor(random_state=MODEL_RANDOM_STATE),
        train_df=train_raw,
        holdout_df=holdout_raw,
        features=ML_FEATURES,
        smooth=MODEL_TARGET_SMOOTH,
    )
    save_model_bundle(bundle_gbm)
except Exception as exc:
    print(f"Model failed: gradient_boosting_log_ratio -> {exc}")

Saved model: gradient_boosting_log_ratio


,Dataset,Rows,Actual_Deaths,Expected_VBT,Predicted_Deaths,Actual_to_Predicted,Actual_to_VBT,Poisson_Deviance,Weighted_AE_Error
0,Train,1494745,"333,551.000000","327,860.433647","318,648.601885",1.046767,1.017357,"278,259.395022",0.492383
1,Holdout,1494745,"333,551.000000","327,860.433647","318,648.601885",1.046767,1.017357,"278,259.395022",0.492383


In [35]:
# ============================================================
# MODEL 6 — HIST GRADIENT BOOSTING MORTALITY RATIO MODEL
# ============================================================

try:
    bundle_hist_gbm = fit_log_ratio_bundle(
        model_key="hist_gradient_boosting_log_ratio",
        estimator=HistGradientBoostingRegressor(
            max_depth=6,
            learning_rate=0.05,
            max_iter=300,
            random_state=MODEL_RANDOM_STATE,
        ),
        train_df=train_raw,
        holdout_df=holdout_raw,
        features=ML_FEATURES,
        smooth=MODEL_TARGET_SMOOTH,
    )
    save_model_bundle(bundle_hist_gbm)
except Exception as exc:
    print(f"Model failed: hist_gradient_boosting_log_ratio -> {exc}")

Saved model: hist_gradient_boosting_log_ratio


,Dataset,Rows,Actual_Deaths,Expected_VBT,Predicted_Deaths,Actual_to_Predicted,Actual_to_VBT,Poisson_Deviance,Weighted_AE_Error
0,Train,1494745,"333,551.000000","327,860.433647","320,696.399274",1.040083,1.017357,"275,432.888885",0.485706
1,Holdout,1494745,"333,551.000000","327,860.433647","320,696.399274",1.040083,1.017357,"275,432.888885",0.485706


In [36]:
# ============================================================
# MODEL 7 — EXTRATREES MORTALITY RATIO MODEL
# ============================================================

try:
    bundle_extratrees = fit_log_ratio_bundle(
        model_key="extratrees_log_ratio",
        estimator=ExtraTreesRegressor(
            n_estimators=400,
            min_samples_leaf=5,
            random_state=MODEL_RANDOM_STATE,
            n_jobs=-1,
        ),
        train_df=train_raw,
        holdout_df=holdout_raw,
        features=ML_FEATURES,
        smooth=MODEL_TARGET_SMOOTH,
    )
    save_model_bundle(bundle_extratrees)
except Exception as exc:
    print(f"Model failed: extratrees_log_ratio -> {exc}")

Saved model: extratrees_log_ratio


,Dataset,Rows,Actual_Deaths,Expected_VBT,Predicted_Deaths,Actual_to_Predicted,Actual_to_VBT,Poisson_Deviance,Weighted_AE_Error
0,Train,1494745,"333,551.000000","327,860.433647","322,201.829328",1.035224,1.017357,"263,690.258066",0.480312
1,Holdout,1494745,"333,551.000000","327,860.433647","322,201.829328",1.035224,1.017357,"263,690.258066",0.480312


In [37]:
# ============================================================
# SESSION 3 MODEL COMPARISON TABLE
# ============================================================

if len(MODEL_ROWS) == 0:
    model_comparison = pd.DataFrame()
    print("No model results are available yet. Run one or more model cells above.")
else:
    model_comparison = pd.DataFrame(MODEL_ROWS).sort_values(
        ["Holdout_Deviance", "Holdout_Weighted_AE_Error"],
        ascending=[True, True],
    ).reset_index(drop=True)

    display(model_comparison)
    export_csv(model_comparison, "session3_model_comparison.csv")

,Model_Key,Model_Type,Feature_Count,Train_Deviance,Holdout_Deviance,Train_A/P,Holdout_A/P,Train_Weighted_AE_Error,Holdout_Weighted_AE_Error
0,random_forest_log_ratio,log_ratio_model,7,"261,046.588670","261,046.588670",1.034669,1.034669,0.480340,0.480340
1,extratrees_log_ratio,log_ratio_model,7,"263,690.258066","263,690.258066",1.035224,1.035224,0.480312,0.480312
2,hist_gradient_boosting_log_ratio,log_ratio_model,7,"275,432.888885","275,432.888885",1.040083,1.040083,0.485706,0.485706
3,poisson_glm_attained,statsmodels_glm,7,"278,118.614037","278,118.614037",1.000000,1.000000,0.517760,0.517760
4,gradient_boosting_log_ratio,log_ratio_model,7,"278,259.395022","278,259.395022",1.046767,1.046767,0.492383,0.492383
5,poisson_glm_core,statsmodels_glm,7,"279,678.905112","279,678.905112",1.000000,1.000000,0.521286,0.521286
6,poisson_regressor,poisson_regressor,7,"280,417.070477","280,417.070477",1.000050,1.000050,0.522657,0.522657


Saved: script\model_outputs\session3_model_comparison.csv


In [38]:
# Session 3 conclusion paragraph
if "model_comparison" not in globals() or len(model_comparison) == 0:
    print("Session 3 conclusion: no model comparison table is available yet. Run at least one model cell above.")
else:
    best_model_key = model_comparison.loc[0, "Model_Key"]
    best_holdout_dev = model_comparison.loc[0, "Holdout_Deviance"]
    best_holdout_ap = model_comparison.loc[0, "Holdout_A/P"]

    session3_conclusion = (
        f"Session 3 conclusion: on a common row-level holdout scoring basis, the best current model is "
        f"{best_model_key}, with holdout Poisson deviance {best_holdout_dev:,.4f} and holdout A/P {best_holdout_ap:.4f}. "
        f"Because every model is evaluated on the same holdout rows, this comparison is now fit for model-selection use."
    )
    print(session3_conclusion)

Session 3 conclusion: on a common row-level holdout scoring basis, the best current model is random_forest_log_ratio, with holdout Poisson deviance 261,046.5887 and holdout A/P 1.0347. Because every model is evaluated on the same holdout rows, this comparison is now fit for model-selection use.


---

# Session 4 · Testing / validation / sheet-style mortality output

This section keeps the true holdout check separate from the final full-data refit, then builds sheet-style mortality results for one segment at a time.

In [39]:
# ============================================================
# REFIT SELECTED MODEL ON FULL SCOPED DATA
# ============================================================

if SELECTED_MODEL_KEY not in MODEL_BUNDLES:
    raise KeyError(
        f"SELECTED_MODEL_KEY={SELECTED_MODEL_KEY!r} was not found in MODEL_BUNDLES. "
        "Run the model cells in Session 3 first."
    )

selected_train_bundle = MODEL_BUNDLES[SELECTED_MODEL_KEY]

if selected_train_bundle["model_type"] == "statsmodels_glm":
    final_bundle = fit_formula_glm_bundle(
        model_key=SELECTED_MODEL_KEY,
        formula=selected_train_bundle["formula"],
        train_df=scoped_df,
        holdout_df=scoped_df,
    )
elif selected_train_bundle["model_type"] == "poisson_regressor":
    final_bundle = fit_poisson_regressor_bundle(
        model_key=SELECTED_MODEL_KEY,
        train_df=scoped_df,
        holdout_df=scoped_df,
        features=selected_train_bundle["features"],
        alpha=selected_train_bundle["result"].alpha,
    )
elif selected_train_bundle["model_type"] == "log_ratio_model":
    final_estimator = selected_train_bundle["result"].__class__(**selected_train_bundle["result"].get_params())
    final_bundle = fit_log_ratio_bundle(
        model_key=SELECTED_MODEL_KEY,
        estimator=final_estimator,
        train_df=scoped_df,
        holdout_df=scoped_df,
        features=selected_train_bundle["features"],
        smooth=selected_train_bundle["smooth"],
    )
else:
    raise ValueError(f"Unsupported selected model type: {selected_train_bundle['model_type']}")

scored_df = score_with_bundle(final_bundle, scoped_df)
scored_df = add_validation_bands(scored_df)

overall_validation = pd.DataFrame([{
    "Rows": len(scored_df),
    "Actual_Deaths": scored_df[ACTUAL_CNT_COL].sum(),
    "Expected_VBT": scored_df[EXPECTED_CNT_COL].sum(),
    "Predicted_Deaths": scored_df["Predicted_Deaths"].sum(),
    "Actual_to_Predicted": safe_divide(scored_df[ACTUAL_CNT_COL].sum(), scored_df["Predicted_Deaths"].sum()),
    "Actual_to_VBT": safe_divide(scored_df[ACTUAL_CNT_COL].sum(), scored_df[EXPECTED_CNT_COL].sum()),
    "Amount_AE_to_VBT": safe_divide(scored_df[ACTUAL_AMT_COL].sum(), scored_df[EXPECTED_AMT_COL].sum()),
}])

display(overall_validation)
export_csv(overall_validation, "session4_overall_validation_full_refit.csv")

,Rows,Actual_Deaths,Expected_VBT,Predicted_Deaths,Actual_to_Predicted,Actual_to_VBT,Amount_AE_to_VBT
0,1494745,333551,"327,860.433647","333,551.000000",1.000000,1.017357,1.077585


Saved: script\model_outputs\session4_overall_validation_full_refit.csv


In [40]:
# ============================================================
# TRUE HOLDOUT CHECK FOR THE SELECTED MODEL
# ============================================================

if true_holdout_available:
    selected_holdout_scored_df = score_with_bundle(selected_train_bundle, holdout_raw)
    selected_holdout_scored_df = add_validation_bands(selected_holdout_scored_df)

    selected_holdout_summary = pd.DataFrame([{
        "Rows": len(selected_holdout_scored_df),
        "Actual_Deaths": selected_holdout_scored_df[ACTUAL_CNT_COL].sum(),
        "Expected_VBT": selected_holdout_scored_df[EXPECTED_CNT_COL].sum(),
        "Predicted_Deaths": selected_holdout_scored_df["Predicted_Deaths"].sum(),
        "Actual_to_Predicted": safe_divide(selected_holdout_scored_df[ACTUAL_CNT_COL].sum(), selected_holdout_scored_df["Predicted_Deaths"].sum()),
        "Actual_to_VBT": safe_divide(selected_holdout_scored_df[ACTUAL_CNT_COL].sum(), selected_holdout_scored_df[EXPECTED_CNT_COL].sum()),
        "Amount_AE_to_VBT": safe_divide(selected_holdout_scored_df[ACTUAL_AMT_COL].sum(), selected_holdout_scored_df[EXPECTED_AMT_COL].sum()),
    }])

    print("This table is the honest out-of-sample holdout check.")
else:
    selected_holdout_scored_df = pd.DataFrame(columns=scored_df.columns)
    selected_holdout_summary = pd.DataFrame([{
        "Rows": len(scored_df),
        "Actual_Deaths": scored_df[ACTUAL_CNT_COL].sum(),
        "Expected_VBT": scored_df[EXPECTED_CNT_COL].sum(),
        "Predicted_Deaths": scored_df["Predicted_Deaths"].sum(),
        "Actual_to_Predicted": safe_divide(scored_df[ACTUAL_CNT_COL].sum(), scored_df["Predicted_Deaths"].sum()),
        "Actual_to_VBT": safe_divide(scored_df[ACTUAL_CNT_COL].sum(), scored_df[EXPECTED_CNT_COL].sum()),
        "Amount_AE_to_VBT": safe_divide(scored_df[ACTUAL_AMT_COL].sum(), scored_df[EXPECTED_AMT_COL].sum()),
    }])

    print("No separate holdout years were available, so the holdout summary falls back to the full scored data.")

display(selected_holdout_summary)
export_csv(selected_holdout_summary, "session4_selected_holdout_summary.csv")

No separate holdout years were available, so the holdout summary falls back to the full scored data.


,Rows,Actual_Deaths,Expected_VBT,Predicted_Deaths,Actual_to_Predicted,Actual_to_VBT,Amount_AE_to_VBT
0,1494745,333551,"327,860.433647","333,551.000000",1.000000,1.017357,1.077585


Saved: script\model_outputs\session4_selected_holdout_summary.csv


In [41]:
# ============================================================
# CORE CALIBRATION TABLES
# ============================================================

diagnostic_df = selected_holdout_scored_df.copy() if len(selected_holdout_scored_df) > 0 else scored_df.copy()
diagnostic_label = "true_holdout" if len(selected_holdout_scored_df) > 0 else "full_refit"

validation_tables = {
    "by_year": calibration_table(diagnostic_df, "Observation_Year"),
    "by_age_ind": calibration_table(diagnostic_df, "Age_Ind"),
    "by_sex": calibration_table(diagnostic_df, "Sex"),
    "by_plan": calibration_table(diagnostic_df, "Insurance_Plan"),
    "by_face_band": calibration_table(diagnostic_df, "Face_Amount_Band"),
    "by_issue_age_band": calibration_table(diagnostic_df, "Issue_Age_Band"),
    "by_attained_age_band": calibration_table(diagnostic_df, "Attained_Age_Band"),
    "by_duration_band": calibration_table(diagnostic_df, "Duration_Band"),
}

for name, table in validation_tables.items():
    print(f"\n=== {name} ({diagnostic_label}) ===")
    display(table.drop(columns=["Flag_Order"]))
    export_csv(table.drop(columns=["Flag_Order"]), f"session4_{diagnostic_label}_{name}.csv")


=== by_year (full_refit) ===


,Observation_Year,Death_Count,ExpDth_VBT2015_Cnt,Death_Claim_Amount,ExpDth_VBT2015_Amt,Policies_Exposed,Predicted_Deaths,Actual_to_Predicted,Actual_to_VBT,Predicted_to_VBT,Amount_AE_to_VBT,Actual_Rate_per_1000,Expected_Rate_per_1000,Predicted_Rate_per_1000,Gap_from_1_on_A/P,Flag
0,2012,40388,"38,994.414445",258853169,"239,071,530.323833","6,537,836.129823","39,823.206518",1.014183,1.035738,1.021254,1.082744,6.177579,5.964422,6.091191,0.014183,OK
1,2013,38937,"37,732.063441",295147011,"280,901,322.011556","8,165,995.769880","39,243.162831",0.992198,1.031934,1.040048,1.050714,4.768188,4.620632,4.805680,0.007802,OK
2,2014,40716,"38,922.054459",328727659,"300,120,184.713004","8,071,923.269631","40,161.274814",1.013812,1.046091,1.031839,1.095320,5.044151,4.821906,4.975428,0.013812,OK
3,2015,41156,"39,154.441337",352339438,"320,864,569.169383","8,079,322.221069","40,202.204190",1.023725,1.051120,1.026760,1.098094,5.093992,4.846253,4.975938,0.023725,OK
4,2016,40463,"39,977.039576",369969338,"342,087,810.722955","8,084,550.454448","40,754.474823",0.992848,1.012156,1.019447,1.081504,5.004978,4.944869,5.041032,0.007152,OK
5,2017,42111,"41,260.405865",390144487,"360,717,025.737170","7,990,801.635941","41,710.210574",1.009609,1.020615,1.010902,1.081580,5.269934,5.163488,5.219778,0.009609,OK
6,2018,45269,"45,503.804212",407078025,"375,192,004.778064","10,029,021.002603","45,596.091790",0.992826,0.994840,1.002028,1.084986,4.513800,4.537213,4.546415,0.007174,OK
7,2019,44511,"46,316.210312",413042413,"393,649,372.913980","9,892,420.681567","46,060.374460",0.966362,0.961024,0.994476,1.049265,4.499505,4.681990,4.656128,0.033638,OK


Saved: script\model_outputs\session4_full_refit_by_year.csv

=== by_age_ind (full_refit) ===


,Age_Ind,Death_Count,ExpDth_VBT2015_Cnt,Death_Claim_Amount,ExpDth_VBT2015_Amt,Policies_Exposed,Predicted_Deaths,Actual_to_Predicted,Actual_to_VBT,Predicted_to_VBT,Amount_AE_to_VBT,Actual_Rate_per_1000,Expected_Rate_per_1000,Predicted_Rate_per_1000,Gap_from_1_on_A/P,Flag
0,ALB,86120,"68,411.633923",1021675965,"799,799,336.832713","34,278,694.595144","86,120.000000",1.000000,1.258850,1.258850,1.277415,2.512348,1.995748,2.512348,0.000000,OK
1,ANB,247431,"259,448.799723",1793625575,"1,812,804,483.537234","32,573,176.569818","247,431.000000",1.000000,0.953679,0.953679,0.989420,7.596158,7.965106,7.596158,0.000000,OK


Saved: script\model_outputs\session4_full_refit_by_age_ind.csv

=== by_sex (full_refit) ===


...

Saved: script\model_outputs\session4_full_refit_by_issue_age_band.csv

=== by_attained_age_band (full_refit) ===


,Attained_Age_Band,Death_Count,ExpDth_VBT2015_Cnt,Death_Claim_Amount,ExpDth_VBT2015_Amt,Policies_Exposed,Predicted_Deaths,Actual_to_Predicted,Actual_to_VBT,Predicted_to_VBT,Amount_AE_to_VBT,Actual_Rate_per_1000,Expected_Rate_per_1000,Predicted_Rate_per_1000,Gap_from_1_on_A/P,Flag
0,"[0, 5)",263,234.744091,20688043,"20,072,411.264863","2,454,406.930894",227.777494,1.154636,1.120369,0.970323,1.030671,0.107154,0.095642,0.092803,0.154636,Review
1,"[5, 10)",321,325.523079,20093055,"26,012,244.545072","4,261,594.992815",340.119431,0.943786,0.986105,1.044840,0.772446,0.075324,0.076385,0.079810,0.056214,OK
2,"[10, 15)",603,550.299438,30189357,"38,073,506.826286","5,163,373.691370",626.040650,0.963196,1.095767,1.137636,0.792923,0.116784,0.106577,0.121246,0.036804,OK
3,"[15, 20)",2428,"2,453.737307",116954159,"152,211,399.603279","5,808,694.216222","3,039.181703",0.798899,0.989511,1.238593,0.768367,0.417994,0.422425,0.523213,0.201101,Review
4,"[20, 25)",4131,"3,260.395597",206014968,"182,047,347.018016","5,431,726.311490","4,398.447385",0.939195,1.267024,1.349053,1.131656,0.760532,0.600250,0.809770,0.060805,OK
5,"[25, 30)",4855,"3,169.400181",217760227,"159,782,361.144070","5,152,963.101377","4,579.517103",1.060155,1.531836,1.444916,1.362855,0.942176,0.615064,0.888715,0.060155,OK
6,"[30, 35)",5864,"3,604.361197",210339049,"144,843,539.803302","4,926,812.179617","5,383.895994",1.089174,1.626918,1.493717,1.452181,1.190222,0.731581,1.092775,0.089174,OK
7,"[35, 40)",7181,"5,253.500175",193701310,"158,932,561.720131","4,719,899.649384","7,869.406684",0.912521,1.366898,1.497936,1.218764,1.521431,1.113053,1.667283,0.087479,OK
8,"[40, 45)",8374,"6,702.693640",167252767,"148,301,137.893590","4,315,572.922487","9,851.325805",0.850038,1.249348,1.469756,1.127792,1.940414,1.553141,2.282739,0.149962,Review
9,"[45, 50)",11850,"8,004.549488",150778987,"113,066,224.615095","4,187,025.772004","11,232.513442",1.054973,1.480408,1.403266,1.333546,2.830171,1.911751,2.682695,0.054973,OK


Saved: script\model_outputs\session4_full_refit_by_attained_age_band.csv

=== by_duration_band (full_refit) ===


,Duration_Band,Death_Count,ExpDth_VBT2015_Cnt,Death_Claim_Amount,ExpDth_VBT2015_Amt,Policies_Exposed,Predicted_Deaths,Actual_to_Predicted,Actual_to_VBT,Predicted_to_VBT,Amount_AE_to_VBT,Actual_Rate_per_1000,Expected_Rate_per_1000,Predicted_Rate_per_1000,Gap_from_1_on_A/P,Flag
0,0-1,168,187.949162,12255658,"16,617,128.730797","1,435,833.892084",162.316452,1.035015,0.893859,0.863619,0.737532,0.117005,0.130899,0.113047,0.035015,OK
1,2-5,863,888.365398,56340078,"75,188,419.125351","5,700,757.542064",895.777495,0.963409,0.971447,1.008344,0.749319,0.151383,0.155833,0.157133,0.036591,OK
2,6-10,1557,"1,318.123081",97711739,"97,442,591.984478","5,749,768.024881","1,484.417243",1.048896,1.181225,1.126160,1.002762,0.270794,0.229248,0.258170,0.048896,OK
3,11-15,2357,"1,863.632132",115366872,"113,518,581.504001","5,339,802.510841","2,332.723619",1.010407,1.264735,1.251708,1.016282,0.441402,0.349008,0.436856,0.010407,OK
4,16-20,3830,"2,924.960827",168990450,"149,889,211.481848","5,243,327.449773","4,057.971303",0.943821,1.309419,1.387359,1.127436,0.730452,0.557844,0.773931,0.056179,OK
5,21-26+,324776,"320,677.403046",2364636743,"2,159,947,887.543472","43,382,381.745319","324,617.793888",1.000487,1.012781,1.012288,1.094766,7.486357,7.391881,7.482710,0.000487,OK


Saved: script\model_outputs\session4_full_refit_by_duration_band.csv


In [ ]:
# ============================================================
# WORST SEGMENTS TO REVIEW
# ============================================================

worst_segments = pd.concat(
    [
        validation_tables["by_age_ind"].assign(Segment_Type="Age_Ind"),
        validation_tables["by_sex"].assign(Segment_Type="Sex"),
        validation_tables["by_plan"].assign(Segment_Type="Insurance_Plan"),
        validation_tables["by_face_band"].assign(Segment_Type="Face_Amount_Band"),
        validation_tables["by_issue_age_band"].assign(Segment_Type="Issue_Age_Band"),
        validation_tables["by_attained_age_band"].assign(Segment_Type="Attained_Age_Band"),
        validation_tables["by_duration_band"].assign(Segment_Type="Duration_Band"),
    ],
    ignore_index=True,
).sort_values(["Flag_Order", "Gap_from_1_on_A/P"], ascending=[False, False]).reset_index(drop=True)

display(worst_segments.drop(columns=["Flag_Order"]).head(20))
export_csv(worst_segments.drop(columns=["Flag_Order"]), "session4_worst_segments.csv")

In [44]:
# ============================================================
# FINAL TESTING SUMMARY TABLE
# ============================================================

summary_points = pd.DataFrame({
    "Item": [
        "Source file",
        "Scoped rows",
        "Selected model",
        "Holdout years",
        "Validation basis",
        "1M face cap applied",
        "Age_Ind filter",
        "Attained_Age_min",
        "Attained_Age_max",
        "Overall A/P (full refit)",
        "Overall A/VBT (full refit)",
        "Holdout A/P",
        "Output Age_Ind",
        "Output Sex",
    ],
    "Value": [
        str(DATA_PATH),
        f"{len(scoped_df):,}",
        SELECTED_MODEL_KEY,
        str(HOLDOUT_YEARS),
        diagnostic_label,
        str(CAP_FACE_AT_1M),
        str(AGE_IND_KEEP),
        str(ATTAINED_AGE_MIN),
        str(ATTAINED_AGE_MAX),
        round(float(overall_validation.loc[0, "Actual_to_Predicted"]), 6),
        round(float(overall_validation.loc[0, "Actual_to_VBT"]), 6),
        round(float(selected_holdout_summary.loc[0, "Actual_to_Predicted"]), 6),
        str(OUTPUT_AGE_IND),
        str(OUTPUT_SEX),
    ]
})

display(summary_points)
export_csv(summary_points, "session4_summary_points.csv")

,Item,Value
0,Source file,script\data\parquet\full.parquet
1,Scoped rows,"1,494,745"
2,Selected model,poisson_glm_core
3,Holdout years,None
4,Validation basis,full_refit
5,1M face cap applied,True
6,Age_Ind filter,None
7,Attained_Age_min,1
8,Attained_Age_max,94
9,Overall A/P (full refit),1.000000


Saved: script\model_outputs\session4_summary_points.csv


In [ ]:
# ============================================================
# SHEET-STYLE MORTALITY OUTPUTS
# ============================================================

modeled_sheet_pred = build_sheet_style_table(
    scored_df,
    value_col="Predicted_Deaths",
    age_ind_value=OUTPUT_AGE_IND,
    sex_value=OUTPUT_SEX,
    select_period=OUTPUT_SELECT_PERIOD,
    ult_duration=OUTPUT_ULT_DURATION,
    duration_cols=OUTPUT_DURATION_COLUMNS,
    rate_per=OUTPUT_RATE_PER,
    issue_age_min=OUTPUT_ISSUE_AGE_MIN,
    issue_age_max=OUTPUT_ISSUE_AGE_MAX,
)

modeled_sheet_actual = build_sheet_style_table(
    scored_df,
    value_col=ACTUAL_CNT_COL,
    age_ind_value=OUTPUT_AGE_IND,
    sex_value=OUTPUT_SEX,
    select_period=OUTPUT_SELECT_PERIOD,
    ult_duration=OUTPUT_ULT_DURATION,
    duration_cols=OUTPUT_DURATION_COLUMNS,
    rate_per=OUTPUT_RATE_PER,
    issue_age_min=OUTPUT_ISSUE_AGE_MIN,
    issue_age_max=OUTPUT_ISSUE_AGE_MAX,
)

modeled_sheet_vbt = build_sheet_style_table(
    scored_df,
    value_col=EXPECTED_CNT_COL,
    age_ind_value=OUTPUT_AGE_IND,
    sex_value=OUTPUT_SEX,
    select_period=OUTPUT_SELECT_PERIOD,
    ult_duration=OUTPUT_ULT_DURATION,
    duration_cols=OUTPUT_DURATION_COLUMNS,
    rate_per=OUTPUT_RATE_PER,
    issue_age_min=OUTPUT_ISSUE_AGE_MIN,
    issue_age_max=OUTPUT_ISSUE_AGE_MAX,
)

print("Predicted mortality table (rate per 1000)")
display(modeled_sheet_pred)

print("\nActual mortality table (rate per 1000)")
display(modeled_sheet_actual)

print("\n2015 VBT expected table on the same data mix (rate per 1000)")
display(modeled_sheet_vbt)

sheet_stub = f"ageind_{OUTPUT_AGE_IND}_sex_{OUTPUT_SEX}"
export_csv(modeled_sheet_pred, f"session4_modeled_sheet_pred_{sheet_stub}.csv")
export_csv(modeled_sheet_actual, f"session4_modeled_sheet_actual_{sheet_stub}.csv")
export_csv(modeled_sheet_vbt, f"session4_modeled_sheet_vbt_{sheet_stub}.csv")

In [46]:
# ============================================================
# OPTIONAL BENCHMARK COMPARISON TO A REFERENCE EXCEL SHEET
# ============================================================

if BENCHMARK_XLSX_PATH is not None and BENCHMARK_SHEET_NAME is not None:
    reference_sheet = read_reference_sheet(Path(BENCHMARK_XLSX_PATH), BENCHMARK_SHEET_NAME, max_issue_age=OUTPUT_ISSUE_AGE_MAX)
    benchmark_comparison = compare_to_reference(modeled_sheet_pred, reference_sheet)

    print("Reference sheet")
    display(reference_sheet)

    print("\nLargest modeled-minus-reference differences")
    display(benchmark_comparison.head(25))

    export_csv(reference_sheet, f"session4_reference_sheet_{sheet_stub}.csv")
    export_csv(benchmark_comparison, f"session4_benchmark_comparison_{sheet_stub}.csv")
else:
    print("No benchmark workbook/sheet supplied. Skip this cell unless you want an explicit table-to-table comparison.")

No benchmark workbook/sheet supplied. Skip this cell unless you want an explicit table-to-table comparison.


In [47]:
# Session 4 conclusion paragraph
n_review = int((worst_segments["Flag"] == "Review").sum())
overall_ap = float(overall_validation.loc[0, "Actual_to_Predicted"])
holdout_ap = float(selected_holdout_summary.loc[0, "Actual_to_Predicted"])

session4_conclusion = (
    f"Session 4 conclusion: the selected model {SELECTED_MODEL_KEY} produces an overall A/P of {overall_ap:.4f} "
    f"on the full-data refit and a holdout A/P of {holdout_ap:.4f} on the honest validation view. "
    f"There are {n_review} segment tables currently flagged for review after applying the credibility floor of "
    f"{MIN_EXPECTED_FOR_REVIEW:.1f} expected deaths. The sheet-style mortality output now gives a direct "
    f"{OUTPUT_SEX} / Age_Ind {OUTPUT_AGE_IND} juvenile rate table with issue ages {OUTPUT_ISSUE_AGE_MIN}-{OUTPUT_ISSUE_AGE_MAX}, "
    f"so the notebook can be used both for model comparison and for report-ready mortality table production."
)

print(session4_conclusion)

Session 4 conclusion: the selected model poisson_glm_core produces an overall A/P of 1.0000 on the full-data refit and a holdout A/P of 1.0000 on the honest validation view. There are 5 segment tables currently flagged for review after applying the credibility floor of 3.0 expected deaths. The sheet-style mortality output now gives a direct M / Age_Ind 0 juvenile rate table with issue ages 0-17, so the notebook can be used both for model comparison and for report-ready mortality table production.


## Suggested working pattern

1. change the **control panel**
2. rerun **Session 1**
3. review the **Session 2 screening tables**
4. update `SHORTLIST_FEATURES`, `ML_FEATURES`, or the GLM formulas if needed
5. run the explicit model cells in **Session 3**
6. set `SELECTED_MODEL_KEY`
7. rerun **Session 4**
8. set `OUTPUT_AGE_IND` and `OUTPUT_SEX` for the sheet you want
9. review the exported **sheet-style mortality tables** before moving anything into a report